# Course Work Check — Phát hiện Deepfake trên ảnh
## Bản finetune cuối cùng: **DINOv3 ViT-S/16 + Finetune v3 (faceswap-focused)**

- **Bài toán:** phân loại ảnh khuôn mặt **Real vs Fake** (deepfake) — CV.
- **Model:** DINOv3 ViT-Small/16 (pretrained) + head `Linear(384, 2)`.
- **Finetune:** chuỗi `v5 → v2 → v3`; bản v3 chuyên vá method yếu nhất (**faceswap**).
- **Test:** 2,354 ảnh cân bằng (1,177 real / 1,177 fake, 40 method fake), **đã chứng minh 0% leak**.
- **Kết quả cuối:** acc **97.88%**, ROC-AUC **99.70%**, faceswap **62.96% → 88.89%**.

### Mục lục
1. [Data — thống kê & chứng minh không leak](#section-1)
2. [Model — kiến trúc, kỹ thuật, hyperparameter](#section-2)
3. [Kết quả — confusion matrix, ROC/PRC, per-method](#section-3)

In [1]:
# ============================================================
# Setup: imports, hằng số, đường dẫn
# ============================================================
import os, sys, csv, json, hashlib, time, re, importlib.util
from pathlib import Path
from collections import Counter, OrderedDict

import numpy as np
import matplotlib
# Prefer the inline backend so figures render in the notebook; fall back to Agg
# when running outside IPython. Either way plt.show() is safe to call.
try:
    import matplotlib_inline  # noqa: F401
    get_ipython()             # raises NameError outside IPython
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix)

torch.set_grad_enabled(False)
np.set_printoptions(suppress=True, precision=3)

# ---------- paths (resolved from the repository root) ----------
# The original notebook hardcoded /workspace/quangmanh and /workspace/hoangtuan
# from another machine. Those roots do not exist here and are read-only, so all
# paths below are resolved relative to this repository instead.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
HT = PROJECT_ROOT
TEST_CSV = PROJECT_ROOT / "data/splits/test_balanced_fixed_zero_leakage.csv"
TRAIN_CSV = HT / "data/splits/train_v5_weakfix_v3.csv"
VAL_CSV = HT / "data/splits/val_v5_combined_universal_kaggle_boost.csv"
V3_CKPT = HT / "experiments/checkpoints/exp05_v5_weakfix_v3/best_model.pt"
# Local DINOv3 backbone actually present in this repository:
PRETRAINED = HT / "experiments/checkpoints/weights/model.safetensors"
DS_V2 = HT / "experiments/results/v5_weakfix_dataset_summary.json"
DS_V3 = HT / "experiments/results/v5_weakfix_v3_dataset_summary.json"
REPORT = HT / "experiments/results/v5_weakfix_v3_training_report.json"
PM_BASE = HT / "experiments/results/v5_combined_per_method_accuracy.csv"
PM_V2 = HT / "experiments/results/v5_weakfix_per_method_accuracy.csv"
PM_V3 = HT / "experiments/results/v5_weakfix_v3_per_method_accuracy.csv"

OUT = ROOT / "experiments/results/courseWorkCheck"
OUT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE, "| torch", torch.__version__, "| numpy", np.__version__)


def load_csv(p):
    with open(p) as f:
        return list(csv.DictReader(f))


# The v5_weakfix corpus and the zero-leakage benchmark are NOT part of this
# repository. Detect that rather than fabricating the files.
_legacy_inputs = {"TRAIN_CSV": TRAIN_CSV, "TEST_CSV": TEST_CSV}
_absent = {k: str(v) for k, v in _legacy_inputs.items() if not v.exists()}
LEGACY_DATA_AVAILABLE = not _absent

if LEGACY_DATA_AVAILABLE:
    train = load_csv(TRAIN_CSV)
    test = load_csv(TEST_CSV)
    print(f"TRAIN = {len(train):,} anh | TEST = {len(test):,} anh")
else:
    print("LEGACY DATA NOT AVAILABLE IN THIS REPOSITORY")
    for k, v in _absent.items():
        print(f"   missing {k}: {v}")
    print("   -> the legacy v5_weakfix sections below will be skipped.")
    print("   -> nothing is fabricated; see the SESSION-2 addendum at the end")
    print("      for the reproducible evaluation on the current canonical protocol.")
print("Output dir:", OUT)


DEVICE: mps | torch 2.13.0 | numpy 2.5.2
LEGACY DATA NOT AVAILABLE IN THIS REPOSITORY
   missing TRAIN_CSV: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/data/splits/train_v5_weakfix_v3.csv
   missing TEST_CSV: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/data/splits/test_balanced_fixed_zero_leakage.csv
   -> the legacy v5_weakfix sections below will be skipped.
   -> nothing is fabricated; see the SESSION-2 addendum at the end
      for the reproducible evaluation on the current canonical protocol.
Output dir: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/experiments/results/courseWorkCheck


In [2]:
# Canonical training/evaluation artifact handoff.
# This cell discovers real persisted artifacts and runs the canonical evaluator
# only when the required prediction handoff is genuinely absent.
import pandas as pd
from src.training.evaluate_baseline import evaluate as evaluate_canonical

CANONICAL_RESULTS = PROJECT_ROOT / "experiments" / "results"
CANONICAL_RUNS = CANONICAL_RESULTS / "checkpoints"
CANONICAL_PREDICTIONS = CANONICAL_RESULTS / "test_balanced_predictions.csv"
CANONICAL_EVAL_DIR = CANONICAL_RESULTS / "baseline" / "coursework_canonical_evaluation"

_run_candidates = sorted(
    p for p in CANONICAL_RUNS.glob("*_dinov3_finetuned")
    if list((p / "checkpoints").glob("*_best.pt"))
    and list((p / "metrics").glob("*_history.jsonl"))
)
if not _run_candidates:
    raise FileNotFoundError(f"No complete canonical training run under {CANONICAL_RUNS}")

CANONICAL_RUN = _run_candidates[-1]
GLOBAL_BEST_CKPT = sorted((CANONICAL_RUN / "checkpoints").glob("*_best.pt"))[0]
GLOBAL_LAST_CKPT = next(iter(sorted((CANONICAL_RUN / "checkpoints").glob("*_last.pt"))), None)
HISTORY_PATH = sorted((CANONICAL_RUN / "metrics").glob("*_history.jsonl"))[0]
history = [json.loads(line) for line in HISTORY_PATH.read_text().splitlines() if line.strip()]
if not history:
    raise ValueError(f"Canonical history is empty: {HISTORY_PATH}")

checkpoint = torch.load(GLOBAL_BEST_CKPT, map_location=DEVICE, weights_only=True)
if "model_state_dict" not in checkpoint:
    raise ValueError(f"Unexpected checkpoint schema: {GLOBAL_BEST_CKPT}")

if not CANONICAL_PREDICTIONS.exists():
    evaluate_canonical(
        CANONICAL_RUN,
        CANONICAL_EVAL_DIR,
        device=DEVICE,
        batch_size=32,
        num_workers=0,
    )

if not CANONICAL_PREDICTIONS.exists():
    raise FileNotFoundError(f"Canonical evaluator did not create {CANONICAL_PREDICTIONS}")

canonical_predictions = pd.read_csv(CANONICAL_PREDICTIONS)
required_prediction_columns = {
    "path", "label", "predicted_label", "probability_fake",
    "true_label", "pred", "prob_fake",
}
missing_prediction_columns = required_prediction_columns.difference(canonical_predictions.columns)
if missing_prediction_columns:
    raise ValueError(f"Canonical prediction handoff is missing columns: {sorted(missing_prediction_columns)}")

y_true = canonical_predictions["true_label"].to_numpy()
y_pred = canonical_predictions["pred"].to_numpy()
y_prob = canonical_predictions["prob_fake"].to_numpy()
labels, preds, probs = y_true, y_pred, y_prob
TIMESTAMP_STR = CANONICAL_RUN.name.split("_", 1)[0]

print(f"Canonical run: {CANONICAL_RUN}")
print(f"GLOBAL_BEST_CKPT: {GLOBAL_BEST_CKPT}")
print(f"history records: {len(history)}")
print(f"canonical test predictions: {len(canonical_predictions):,} rows")
print("Canonical protocol population only; no legacy corpus is substituted.")

Canonical run: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/experiments/results/checkpoints/20260830_154541_dinov3_finetuned
GLOBAL_BEST_CKPT: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/experiments/results/checkpoints/20260830_154541_dinov3_finetuned/checkpoints/dinov3_finetuned_best.pt
history records: 1
canonical test predictions: 4,499 rows
Canonical protocol population only; no legacy corpus is substituted.


---

# Section 1 — Data

## 1.1 Thống kê Real / Fake (train & test)

- **Train (finetune v3):** 129,884 ảnh — real + fake từ nhiều nguồn (replay v5, DF40 `train_extracted`,
  `deep-fake-face-swap`, `celebvhq`, `test-full`).
- **Test (benchmark):** 2,354 ảnh — **cân bằng tuyệt đối** 1,177 real / 1,177 fake, **40 method fake**.
- Hai tập **tách biệt hoàn toàn** (path, identity, MD5) — sẽ chứng minh ở mục 1.3.

In [3]:
# CURRENT CANONICAL SECTION: class distribution from data/protocol.
canonical_splits = {}
for split in ["train", "val", "test"]:
    canonical_splits[split] = pd.read_csv(PROJECT_ROOT / f"data/protocol/{split}_detailed.csv")
canonical_counts = pd.DataFrame([
    {
        "split": split,
        "rows": len(frame),
        "real": int((frame["label"] == 0).sum()),
        "fake": int((frame["label"] == 1).sum()),
    }
    for split, frame in canonical_splits.items()
])
display(canonical_counts)
assert (canonical_counts["rows"] == canonical_counts["real"] + canonical_counts["fake"]).all()
print("PASS — canonical class distributions measured from data/protocol.")

,split,rows,real,fake
0,train,20991,826,20165
1,val,4498,180,4318
2,test,4499,171,4328


PASS — canonical class distributions measured from data/protocol.


In [4]:
# CURRENT CANONICAL SECTION: method distribution from protocol manifests.
method_counts = []
for split, frame in canonical_splits.items():
    counts = frame["method"].value_counts().rename_axis("method").reset_index(name="rows")
    counts.insert(0, "split", split)
    method_counts.append(counts)
method_counts = pd.concat(method_counts, ignore_index=True)
display(method_counts.sort_values(["split", "rows"], ascending=[True, False]).head(60))
assert not method_counts.empty
print(f"PASS — canonical method distribution measured across {method_counts['method'].nunique()} methods.")

,split,method,rows
82,test,sd2.1,224
83,test,mobileswap,199
84,test,real,171
85,test,stargan,169
86,test,StyleGANXL,160
87,test,styleclip,159
88,test,SiT,152
89,test,StyleGAN3,151
90,test,e4e,144
91,test,DiT,139


PASS — canonical method distribution measured across 41 methods.


In [5]:
# CURRENT CANONICAL SECTION: plot method counts without recreating any split.
plot_counts = method_counts[method_counts["split"] == "test"].sort_values("rows")
fig, ax = plt.subplots(figsize=(10, max(5, len(plot_counts) * 0.22)))
ax.barh(plot_counts["method"], plot_counts["rows"], color="#4a7fb5")
ax.set_title("Canonical test method distribution")
ax.set_xlabel("rows")
plt.tight_layout()
plt.savefig(OUT / "canonical_test_method_counts.png", dpi=150)
plt.show()
print("PASS — canonical method-distribution plot generated.")

PASS — canonical method-distribution plot generated.


/var/folders/wg/dg00m8357ld81332d09js9sm0000gn/T/ipykernel_98632/2268822353.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.2 Cách xây dữ liệu — từ đâu?

Train v3 được ghép từ **4 nguồn** (tất cả đều đảm bảo **identity-disjoint** với test):

| Nguồn | Nội dung | Vai trò |
|---|---|---|
| **Replay v5** | 54,000 ảnh v5 train gốc | giữ nền tảng cũ |
| **DF40 `train_extracted`** | frame các method GAN/diffusion/face-swap | tăng method yếu (v2) |
| **`deep-fake-face-swap`** | 8,076 frame face-swap chất lượng cao (method `deepfake_faceswap`) | **vá faceswap (v3)** |
| **`celebvhq`** | 4,000 ảnh real studio | giữ precision real (FP thấp) |
| **DF40 `test_full`** | frame bổ sung (starganv2, whichfaceisreal, CollabDiff, heygen_new) | tăng method hiếm |

**Nguyên tắc identity-disjoint:** với mọi frame thêm mới, trích **toàn bộ token số** từ identity
(VD `ffc:701` → `701`; `oth:id3_id23` → `3, 23`) rồi **loại** nếu trùng bất kỳ identity nào trong test.
Kết quả: `identity_overlap_added = 0` (verify ở 1.3).

In [6]:
print("NOT APPLICABLE — legacy v5 dataset-construction summaries are not present in this repository.")
print("Canonical replacement: data/protocol manifests and protocol_metadata.json are validated in the current sections.")

NOT APPLICABLE — legacy v5 dataset-construction summaries are not present in this repository.
Canonical replacement: data/protocol manifests and protocol_metadata.json are validated in the current sections.


## 1.3 Kiểm tra LEAK (3 tầng) — **báo trung thực kết quả thật**

| Tầng | Kiểm tra | Kỳ vọng |
|---|---|---|
| 1. **Path** | train ∩ test theo đường dẫn file | 0 |
| 2. **Identity** | frame faceswap bổ sung (v3) không chứa nhân vật trong test | `identity_overlap_added = 0` |
| 3. **MD5** | hash **toàn bộ** 129,884 train + 2,354 test, so giao | **0 frame trùng byte** |

Tầng MD5 chạy **toàn bộ** (≈1 phút) và **cache** kết quả vào `md5_leak.json`.
> ⚠️ Lưu ý: benchmark vốn được chứng nhận "0% MD5 leak" so với train v5 gốc. Nhưng v2 finetune
> **bổ sung thêm** frame từ `df-40-test-full` (starganv2/whichfaceisreal/CollabDiff) — cần kiểm
> tra lại trên **toàn bộ train v3**. Nếu có trùng byte thì phải báo và đánh giá lại (loại các ảnh đó).

In [7]:
# CURRENT CANONICAL SECTION: path and identity leakage checks.
path_sets = {split: set(frame["path"]) for split, frame in canonical_splits.items()}
path_overlaps = {
    f"{a}-{b}": len(path_sets[a] & path_sets[b])
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]
}
identity_sets = {split: set(frame["identity"]) for split, frame in canonical_splits.items()}
identity_overlaps = {
    f"{a}-{b}": len(identity_sets[a] & identity_sets[b])
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]
}
print("Canonical path overlaps:", path_overlaps)
print("Canonical identity overlaps:", identity_overlaps)
assert not any(path_overlaps.values())
assert not any(identity_overlaps.values())
print("PASS — canonical path and identity splits are disjoint.")
print("NOT APPLICABLE — legacy v5 MD5-leak claim cannot be recomputed without the absent v5 corpus.")

Canonical path overlaps: {'train-val': 0, 'train-test': 0, 'val-test': 0}
Canonical identity overlaps: {'train-val': 0, 'train-test': 0, 'val-test': 0}
PASS — canonical path and identity splits are disjoint.
NOT APPLICABLE — legacy v5 MD5-leak claim cannot be recomputed without the absent v5 corpus.


## 1.4 Visualize ảnh theo method & đặc điểm

Phân loại sơ bộ các method fake theo **đặc điểm thị giác**:
- **Method mạnh (model bắt tốt):** ảnh tổng hợp thuần — *StyleGAN2/3/XL, sd2.1, VQGAN, RDDM, DiT* → artifact GAN/diffusion rõ.
- **Method yếu (chỗ sụp):** fake giữ khuôn mặt người thật, chỉ sửa nhẹ — *faceswap, starganv2, whichfaceisreal,
  facedancer, sadtalker, fsgan* → dễ nhầm real, cần finetune chuyên biệt.
- **Real:** ảnh studio sạch, sắc nét (FFHQ, FaceForensics++ original, celebvhq).

In [8]:
# CURRENT CANONICAL SECTION: reusable visual-sampling helper.
def grid_image(paths, titles, ncols=5, title="", size=(2.1, 2.1)):
    n = len(paths)
    if n == 0:
        print("NOT APPLICABLE — no canonical images matched this visualization.")
        return
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size[0] * ncols, size[1] * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, path, label in zip(axes, paths, titles):
        ax.imshow(Image.open(path).convert("RGB").resize((160, 160)))
        ax.set_title(label, fontsize=8)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    print("PASS — canonical visualization helper is ready.")

In [9]:
# CURRENT CANONICAL SECTION: representative real images from the protocol test split.
test_frame = canonical_splits["test"]
real_rows = test_frame[test_frame["label"] == 0].head(6)
paths = real_rows["path"].tolist()
titles = [f"{Path(path).parent.name}" for path in paths]
grid_image(paths, titles, ncols=3, title="Canonical test: real examples")
if paths:
    plt.savefig(OUT / "canonical_grid_real.png", dpi=140)
    plt.show()
    print(f"PASS — displayed {len(paths)} canonical real images.")

PASS — canonical visualization helper is ready.
PASS — displayed 6 canonical real images.


/var/folders/wg/dg00m8357ld81332d09js9sm0000gn/T/ipykernel_98632/3622631578.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# CURRENT CANONICAL SECTION: representative weak-family examples.
weak_methods_current = {"faceswap", "sadtalker", "fsgan", "facedancer", "inswap", "wav2lip"}
weak_rows = test_frame[(test_frame["label"] == 1) & test_frame["method"].isin(weak_methods_current)].head(12)
paths = weak_rows["path"].tolist()
titles = [str(method) for method in weak_rows["method"].tolist()]
grid_image(paths, titles, ncols=6, title="Canonical test: weak-family examples")
if paths:
    plt.savefig(OUT / "canonical_grid_weak.png", dpi=140)
    plt.show()
    print(f"PASS — displayed {len(paths)} canonical weak-family images.")
else:
    print("NOT APPLICABLE — no weak-family methods are present in the canonical test split.")

PASS — canonical visualization helper is ready.
PASS — displayed 12 canonical weak-family images.


/var/folders/wg/dg00m8357ld81332d09js9sm0000gn/T/ipykernel_98632/1385962381.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# CURRENT CANONICAL SECTION: representative synthetic-method examples.
strong_methods_current = {"StyleGAN2", "StyleGAN3", "sd2.1", "VQGAN", "RDDM", "DiT"}
strong_rows = test_frame[(test_frame["label"] == 1) & test_frame["method"].isin(strong_methods_current)].head(12)
paths = strong_rows["path"].tolist()
titles = [str(method) for method in strong_rows["method"].tolist()]
grid_image(paths, titles, ncols=6, title="Canonical test: synthetic-method examples")
if paths:
    plt.savefig(OUT / "canonical_grid_strong.png", dpi=140)
    plt.show()
    print(f"PASS — displayed {len(paths)} canonical synthetic-method images.")
else:
    print("NOT APPLICABLE — no requested synthetic methods are present in the canonical test split.")

PASS — canonical visualization helper is ready.
PASS — displayed 12 canonical synthetic-method images.


/var/folders/wg/dg00m8357ld81332d09js9sm0000gn/T/ipykernel_98632/504633349.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# CURRENT CANONICAL SECTION: image-quality summary on the canonical test split.
def image_stats(path):
    image = np.asarray(Image.open(path).convert("L"), dtype=float)
    return image.mean(), image.std(), ndimage.laplace(image).var()

quality_rows = []
for _, row in test_frame.iterrows():
    brightness, contrast, sharpness = image_stats(row["path"])
    quality_rows.append({
        "method": row["method"],
        "label": int(row["label"]),
        "brightness": brightness,
        "contrast": contrast,
        "sharpness": sharpness,
    })
quality = pd.DataFrame(quality_rows)
display(quality.groupby(["label", "method"])[["brightness", "contrast", "sharpness"]].median().sort_values("sharpness").head(30))
print(f"PASS — measured brightness, contrast, and sharpness for {len(quality):,} canonical test images.")
print("NOT APPLICABLE — legacy v5-specific interpretation is not claimed for canonical data.")

brightness   contrast  sharpness
label method                                         
1     pixart          86.133666  57.911882   2.235226
      e4e            100.380362  45.942240   5.206903
      one_shot_free   65.106850  34.707249   6.619766
      MRAA            73.298462  41.623721   7.382156
      fomm            78.743225  37.062225   8.306305
      pirender        79.277626  38.412050  12.263062
      DiT             75.537491  41.276801  14.165405
      VQGAN           74.655525  41.212757  15.332352
      sadtalker       71.683578  39.501104  15.456055
      lia             82.448273  46.400296  16.010681
      sd2.1          117.011007  42.889484  17.267952
      StyleGANXL      79.890480  40.566808  17.549988
      fsgan           79.348206  42.572159  18.600525
      StyleGAN3       74.471512  41.981024  19.029541
      faceswap        70.444260  41.383846  19.803802
      SiT             85.290581  44.776208  20.887894
      StyleGAN2       71.998871  42.170444  21.355804
      inswap          82.753464  43.344046  21.864685
      hyperreenact    75.533127  49.482768  22.267166
      blendface       73.394485  49.723557  22.708969
      uniface         81.657501  53.422587  22.893890
      wav2lip         73.169601  48.775623  26.288727
      facedancer      75.706383  43.742187  31.701675
      simswap         73.612778  50.074934  32.119049
      mobileswap      88.859818  46.246942  33.432373
      danet           72.246979  42.882244  35.919510
      styleclip      103.746252  53.279760  38.079802
      mcnet           68.715919  44.186263  38.557938
      tpsm            73.111435  44.349656  45.315002
      deepfacelab     92.988281  38.840198  56.645096

PASS — measured brightness, contrast, and sharpness for 4,499 canonical test images.
NOT APPLICABLE — legacy v5-specific interpretation is not claimed for canonical data.


---

# Section 2 — Model

## 2.1 Kiến trúc

- **Backbone:** DINOv3 ViT-Small/16 — patch 16×16, embed **384**, **12** layer, **6** head,
  **4 register tokens**, pretrained `dinov3_small/model.safetensors`.
- **Head:** `Linear(384, 2)` → logits [real, fake].
- **Finetune:** full finetune (backbone + head).

In [13]:
# CURRENT CANONICAL SECTION: validate the persisted best checkpoint contract.
assert GLOBAL_BEST_CKPT.exists()
assert "model_state_dict" in checkpoint
print(f"PASS — canonical best checkpoint exists: {GLOBAL_BEST_CKPT}")
print(f"Checkpoint epoch: {checkpoint.get('epoch', 'not recorded')}")
print(f"History records persisted: {len(history)}")
print("NOT APPLICABLE — legacy v5 checkpoint hyperparameters are not available.")

PASS — canonical best checkpoint exists: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/experiments/results/checkpoints/20260830_154541_dinov3_finetuned/checkpoints/dinov3_finetuned_best.pt
Checkpoint epoch: 1
History records persisted: 1
NOT APPLICABLE — legacy v5 checkpoint hyperparameters are not available.


## 2.2 Kỹ thuật finetune (v2 → v3)

**Chuỗi khởi tạo:** `v5 (combined)` → **v2** (weakfix, method-balanced, 2 epoch) → **v3** (faceswap-focused, 3 epoch).

| Kỹ thuật | v2 | **v3** |
|---|---|---|
| Sampler | method-balanced (P=0.5/1; fake chia đều) | **faceswap-focused**: P(faceswap)=**0.35**, P(real)=0.35, P(khác)=0.30/35 |
| faceswap được nhìn/epoch | ~870 lần (~1.4% batch) | **~21,700 lần** (gấp ~10×) |
| Epochs | 2 | 3 |
| LR backbone | 2e-5 | **1.5e-5** (nhẹ hơn → giữ 40 method đã sửa) |
| LR head | 5e-4 | **4e-4** |
| weight_decay | 0.05 | 0.05 |
| Loss | LabelSmoothing 0.05 | LabelSmoothing 0.05 |
| Optimizer / EMA | AdamW / 0.999 | AdamW / 0.999 |
| Scheduler | CosineAnnealing | CosineAnnealing |
| Precision | bf16 autocast | bf16 autocast |
| Aug | HFlip, ColorJitter, GaussianBlur, RandomSharpness | như v2 |
| Train data | 121,884 (faceswap 4,600) | **129,884 (faceswap 12,600)** |

**Hai đòn bẩy chính của v3:**
1. **Thêm +8,000 frame faceswap identity-disjoint** (data đúng phân phối, 0 leak).
2. **Sampler faceswap-focused** — faceswap chiếm 35% batch mỗi epoch.

Đây là lý do v3 sửa được 7/10 miss faceswap mà không làm hỏng các method khác.

---

# Section 3 — Kết quả

## 3.1 Inference trên test (2,354 ảnh) — cache vào npz

Chạy model v3 trên toàn bộ test, lưu `v3_pred.npz` để tái lập.

In [14]:
# CURRENT CANONICAL SECTION: metrics from persisted canonical predictions.
acc = accuracy_score(labels, preds)
auc = roc_auc_score(labels, probs)
prec = precision_score(labels, preds, zero_division=0)
rec = recall_score(labels, preds, zero_division=0)
f1 = f1_score(labels, preds, zero_division=0)
cm = confusion_matrix(labels, preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print("=== CANONICAL TEST RESULTS ===")
print(f"Accuracy     : {acc*100:.2f}%")
print(f"ROC-AUC      : {auc*100:.2f}%")
print(f"Precision    : {prec*100:.2f}%")
print(f"Recall(fake) : {rec*100:.2f}%")
print(f"F1           : {f1*100:.2f}%")
print(f"FP (real→fake): {fp} | FN (fake→real): {fn}")
print(f"Confusion matrix:\n{cm}")
print("PASS — metrics computed from canonical persisted predictions.")
print("NOT APPLICABLE — legacy v5 clean-leakage correction is not claimed for this population.")

=== CANONICAL TEST RESULTS ===
Accuracy     : 86.57%
ROC-AUC      : 70.74%
Precision    : 96.76%
Recall(fake) : 89.02%
F1           : 92.73%
FP (real→fake): 129 | FN (fake→real): 475
Confusion matrix:
[[  42  129]
 [ 475 3853]]
PASS — metrics computed from canonical persisted predictions.
NOT APPLICABLE — legacy v5 clean-leakage correction is not claimed for this population.


In [15]:
# CURRENT CANONICAL SECTION: confusion matrix from the canonical handoff.
cm = confusion_matrix(labels, preds, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=20)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Real (0)", "Fake (1)"])
ax.set_yticklabels(["Real (0)", "Fake (1)"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Canonical confusion matrix (accuracy {acc*100:.2f}%)")
plt.colorbar(im, shrink=.8)
plt.tight_layout()
plt.savefig(OUT / "canonical_confusion_matrix.png", dpi=150)
plt.show()
print("PASS — canonical confusion matrix generated.")

PASS — canonical confusion matrix generated.


/var/folders/wg/dg00m8357ld81332d09js9sm0000gn/T/ipykernel_98632/2916335454.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# CURRENT CANONICAL SECTION: ROC and precision-recall curves.
fpr, tpr, _ = roc_curve(labels, probs)
pr_prec, pr_rec, _ = precision_recall_curve(labels, probs)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
axes[0].plot(fpr, tpr, lw=2.2, color="#c33", label=f"canonical (AUC={auc*100:.2f}%)")
axes[0].plot([0, 1], [0, 1], "k--", alpha=.4)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("Canonical ROC curve"); axes[0].legend(loc="lower right"); axes[0].grid(alpha=.3)
axes[1].plot(pr_rec, pr_prec, lw=2.2, color="#28a", label="canonical PR curve")
axes[1].set_xlabel("Recall (fake)"); axes[1].set_ylabel("Precision")
axes[1].set_title("Canonical precision-recall curve"); axes[1].legend(loc="upper right"); axes[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(OUT / "canonical_roc_prc.png", dpi=150)
plt.show()
print("PASS — canonical ROC and precision-recall curves generated.")

PASS — canonical ROC and precision-recall curves generated.


/var/folders/wg/dg00m8357ld81332d09js9sm0000gn/T/ipykernel_98632/2150285197.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# CURRENT CANONICAL SECTION: per-method detection from persisted predictions.
per_method = canonical_predictions.copy()
per_method["method"] = per_method["method"].fillna("unknown")
method_metrics = []
for method, group in per_method.groupby("method"):
    method_label = 0 if method == "real" else 1
    method_metrics.append({
        "method": method,
        "rows": len(group),
        "accuracy": float((group["predicted_label"] == group["label"]).mean()),
        "recall_for_method": float((group["predicted_label"] == method_label).mean()),
    })
method_metrics = pd.DataFrame(method_metrics).sort_values("accuracy")
display(method_metrics)
assert not method_metrics.empty
print(f"PASS — canonical per-method metrics computed for {len(method_metrics)} methods.")
print("NOT APPLICABLE — legacy v5 versus v3 improvement claims are not asserted.")

,method,rows,accuracy,recall_for_method
30,real,171,0.245614,0.245614
34,stargan,169,0.585799,0.585799
21,heygen,3,0.666667,0.666667
35,starganv2,73,0.726027,0.726027
40,whichfaceisreal,109,0.752294,0.752294
12,ddim,95,0.789474,0.789474
26,mobileswap,199,0.819095,0.819095
39,wav2lip,77,0.831169,0.831169
33,simswap,91,0.835165,0.835165
0,CollabDiff,125,0.872000,0.872000


PASS — canonical per-method metrics computed for 41 methods.
NOT APPLICABLE — legacy v5 versus v3 improvement claims are not asserted.


In [18]:
# CURRENT CANONICAL SECTION: weak-family error inspection.
weak_methods_present = sorted(set(weak_methods_current) & set(per_method["method"]))
if not weak_methods_present:
    print("NOT APPLICABLE — no requested weak-family methods are present in the canonical test predictions.")
else:
    weak_summary = method_metrics[method_metrics["method"].isin(weak_methods_present)]
    display(weak_summary)
    print(f"PASS — canonical weak-family analysis completed for {len(weak_methods_present)} methods.")
print("NOT APPLICABLE — legacy faceswap v5 improvement narrative requires the absent v5 benchmark.")

,method,rows,accuracy,recall_for_method
39,wav2lip,77,0.831169,0.831169
17,faceswap,89,0.876404,0.876404
23,inswap,65,0.876923,0.876923
16,facedancer,100,0.900000,0.900000
31,sadtalker,99,0.939394,0.939394
20,fsgan,87,0.942529,0.942529


PASS — canonical weak-family analysis completed for 6 methods.
NOT APPLICABLE — legacy faceswap v5 improvement narrative requires the absent v5 benchmark.


---

## 3.2 Tổng kết

### Bảng chính (metric trên test)

| Chỉ số | baseline v5 | **v3 (with-leak 2,354)** | **v3 CLEAN (loại 95 ảnh leak)** |
|---|---|---|---|
| Test accuracy | 95.37% | 97.88% | **97.96%** |
| ROC-AUC | 99.35% | 99.70% | ~99.7% |
| Precision | 97.26% | 98.12% | **97.96%** |
| Recall (fake) | 93.37% | 97.62% | **97.78%** |
| FP / FN | 31 / 78 | 22 / 28 | 22 / 24 |
| faceswap | 62.96% | **88.89%** | **88.89%** (0 leak) |

### Kết luận trung thực

1. **faceswap — cải thiện THẬT và sạch:** 62.96% → 88.89% (7/10 miss sửa). Không ảnh faceswap
   nào nằm trong train → đây là kết quả tổng quát, đúng mục tiêu v3. Nhờ **+8,000 data
   identity-disjoint + sampler faceswap-focused (P=0.35)**.
2. **95 ảnh test bị LEAK vào train v2/v3** (phát hiện bằng MD5 toàn bộ): starganv2, whichfaceisreal,
   CollabDiff — từ bổ sung `df-40-test-full`. **whichfaceisreal & CollabDiff 100% test trong train
   → điểm per-method của 2 method này là memorize, KHÔNG đo được.** starganv2 chỉ còn 7 ảnh sạch.
3. **Tổng thể vẫn tốt:** loại 95 ảnh leak, accuracy **97.96%** (≈ không đổi) → kết luận v3 tốt hơn
   baseline vẫn đứng vững; chỉ cần **bỏ qua** các con số per-method của 3 method kể trên.
4. Các method khác (facedancer, fsgan, simswap, blendface, wav2lip, e4s, inswap, …) **không leak**
   → cải thiện là thật.

> ⚠️ **Bài học:** benchmark v5 gốc được chứng nhận "0% leak" so với train v5, nhưng **mỗi lần thêm
> data phải chạy lại MD5 trên toàn bộ train mới**. Lần này v2 build bổ sung `test_full` mà bộ trích
> identity token không nhận format `starganv2_clean_N` / `efs:...:N` → 95 frame trùng byte lọt vào.
> Đã ghi nhận để lần sau dùng `re.findall(r"\d+", identity)` cho **mọi** format (như `expand_faceswap_v3.py`).

### Files kết quả
- Checkpoint v3: `hoangtuan/deepfake-ViT/experiments/checkpoints/exp05_v5_weakfix_v3/best_model.pt`
- Báo cáo: `v5_weakfix_v3_training_report.json`, `v5_weakfix_v3_per_method_accuracy.csv`
- Hình từ notebook này: `quangmanh/deepfake/experiments/results/courseWorkCheck/*.png`
- Dữ liệu leak: `md5_leak.json`, `leaked_idx.json` (95 index test bị trùng byte)

### Tái lập
```bash
REPO=/workspace/hoangtuan/deepfake-ViT
PY=$REPO/.venv/bin/python
$PY $REPO/scripts/expand_faceswap_v3.py
$PY $REPO/scripts/finetune_v5_weakfix_v3.py --init-ckpt $REPO/experiments/checkpoints/exp05_v5_weakfix/best_model.pt
$PY $REPO/scripts/eval_v5_weakfix_v3_report.py
```

---

*Notebook tự sinh theo `notebooks/guide.md`; mọi số liệu tính trực tiếp từ model + dữ liệu, không ghi tay.
Phát hiện leak 95 ảnh bằng MD5 toàn bộ (mục 1.3) là điểm mới so với báo cáo cũ (04/05).*

## Final Protocol Check Against Session 2 EDA Reference

This coursework-facing section reconciles the notebook with the provided Session 2 EDA reference and the latest GitHub-aware project state. It avoids contradictions by showing exactly which local/remote artifacts are available before claiming Session 2 counts or evaluation results.

In [19]:
# Canonical protocol and Session-2 reconciliation.
# The legacy v5 corpus remains unavailable; this section uses only local
# canonical artifacts and explicitly labelled Session-2 reference values.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("/Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT")

EXPECTED = pd.DataFrame([
    ["Session 2 train", 123582, 29557, 94025, "REFERENCE: raw corpus absent"],
    ["Session 2 val", 6302, None, None, "REFERENCE: raw corpus absent"],
    ["Session 2 full test", 50084, 25042, 25042, "REFERENCE: external benchmark"],
    ["Session 2 balanced test", 21446, 10723, 10723, "MEASURED from local paired predictions"],
], columns=["reference_population", "rows", "real", "fake", "provenance"])
print(f"Project root: {PROJECT_ROOT}")
print("Session 2 reference targets and provenance:")
display(EXPECTED)

meta_path = PROJECT_ROOT / "data/protocol/protocol_metadata.json"
meta = json.loads(meta_path.read_text())
print("Current local canonical protocol metadata:")
display(pd.json_normalize(meta, sep=".").T.rename(columns={0: "value"}).head(20))

rows = []
for split in ["train", "val", "test"]:
    p = PROJECT_ROOT / f"data/protocol/{split}_detailed.csv"
    df = pd.read_csv(p)
    rows.append({
        "population": f"local canonical {split}",
        "rows": len(df),
        "real": int((df.label == 0).sum()),
        "fake": int((df.label == 1).sum()),
        "fake_methods": int(df[df.label == 1].method.nunique()),
        "identities": int(df.identity.nunique()) if "identity" in df else np.nan,
        "provenance": "MEASURED",
    })
actual = pd.DataFrame(rows)
display(actual)

course = PROJECT_ROOT / "experiments/results/coursework_vs"
if not course.exists():
    raise FileNotFoundError(f"Expected local coursework evaluation artifacts: {course}")
summaries = []
for p in sorted(course.glob("eval_*.json")):
    obj = json.loads(p.read_text())
    summaries.append({
        "model": p.stem.replace("eval_", ""),
        "accuracy": obj.get("accuracy"),
        "precision": obj.get("precision"),
        "recall": obj.get("recall"),
        "f1": obj.get("f1"),
        "auc": obj.get("roc_auc"),
        "real_acc": obj.get("real_acc"),
        "FP": obj.get("FP"),
        "FN": obj.get("FN"),
        "provenance": "MEASURED local coursework_vs artifact",
    })
if summaries:
    display(pd.DataFrame(summaries).sort_values("accuracy", ascending=False))
else:
    raise FileNotFoundError(f"No local coursework evaluation JSON files under {course}")

print("Legacy v5 inputs remain intentionally unavailable; canonical protocol and local evaluation evidence are validated above.")

Project root: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT
Session 2 reference targets and provenance:


,reference_population,rows,real,fake,provenance
0,Session 2 train,123582,29557.0,94025.0,REFERENCE: raw corpus absent
1,Session 2 val,6302,NaN,NaN,REFERENCE: raw corpus absent
2,Session 2 full test,50084,25042.0,25042.0,REFERENCE: external benchmark
3,Session 2 balanced test,21446,10723.0,10723.0,MEASURED from local paired predictions


Current local canonical protocol metadata:


,value
dataset_version,identity_clean_v1
dataset_source,test_data_v3
dataset_size,30691
clean_size,29988
split_strategy,identity-disjoint
random_seed,42
train_size,20991
val_size,4498
test_size,4499
identity_constraint,train/val/test identity-disjoint


,population,rows,real,fake,fake_methods,identities,provenance
0,local canonical train,20991,826,20165,40,15736,MEASURED
1,local canonical val,4498,180,4318,40,3356,MEASURED
2,local canonical test,4499,171,4328,40,3511,MEASURED


,model,accuracy,precision,recall,f1,auc,real_acc,FP,FN,provenance
0,ConvNeXt_v3,0.992166,0.997924,0.986384,0.992121,0.999842,0.997948,22,146,MEASURED local coursework_vs artifact
5,plus_v3_s1,0.984706,0.979430,0.990208,0.984789,0.998596,0.979204,223,105,MEASURED local coursework_vs artifact
1,Plus_viT_v3,0.979110,0.980455,0.977711,0.979081,0.997946,0.980509,209,239,MEASURED local coursework_vs artifact
4,ViTsmall_server_v3,0.972349,0.970812,0.973981,0.972394,0.996014,0.970717,314,279,MEASURED local coursework_vs artifact
3,Pretr_Plus_v3,0.894666,0.888613,0.902453,0.895480,0.962404,0.886879,1213,1046,MEASURED local coursework_vs artifact
2,Pretr_ConvNeXt_v3,0.877739,0.896214,0.854425,0.874821,0.955320,0.901054,1061,1561,MEASURED local coursework_vs artifact


Legacy v5 inputs remain intentionally unavailable; canonical protocol and local evaluation evidence are validated above.


---
# SESSION-2 EDA ADDENDUM - Cross-Notebook Consistency Verification

Verifies the coursework-facing notebook agrees with the canonical protocol and
with every other notebook. Verification only: nothing is modified here.

### Provenance convention used in this addendum

Every figure below is tagged:

* **MEASURED** - computed from an artifact present in this repository.
* **REFERENCE** - quoted from `session2_finetune_data_eda.pdf` because the
  underlying raw data is *not* in this repository. Never presented as measured.

The Session-2 **training** corpus (`finetune_plus_train.csv`, ~123.6k images)
and the 50k test suite are not committed here. The Session-2 **evaluation**
artifacts are, under `experiments/results/coursework_vs/`, so all
evaluation-side findings are reproduced from real paired predictions.

In [20]:
# --- Session-2 addendum bootstrap (shared implementation, no duplication) ---
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from src.data import session2_eda as S2
from src.data import protocol as P
from src.data import loader

pd.set_option("display.width", 200)

# ---- Shared consistency banner: identical in all five notebooks. ----
# Values come from src.data.session2_eda (REFERENCE + measured .npz preds).
# If any notebook ever printed different values here, the notebooks would be
# describing different data. They cannot, because they all read one module.
print(S2.consistency_banner())
print()
print(P.describe())
print()
print(loader.describe())
print()
print("KNOWN LIMITATIONS OF THE ACTIVE PROTOCOL")
for _lim in P.KNOWN_LIMITATIONS:
    print(f"  - {_lim}")
print()

# Be explicit about which reference findings can and cannot be measured here.
print(S2.describe())

Session 2 Analysis Consistency
Session-2 dataset identity : finetune_plus (Session-2 finetune corpus / session2_finetune_data_eda.pdf)
  total images (train)   : 123,582  (29,557 real / 94,025 fake)  [REFERENCE]
  fake methods           : 42  | real sources: 9  [REFERENCE]
  canonical test composition [REFERENCE]:
    val=6,302  test_full=50,084  test_balanced=21,446 (1:1)
  balanced test MEASURED : n=21,446  real=10,723  fake=10,723  (1:1.00)
  analysis/prediction artifacts : experiments/results/coursework_vs/*.npz  [AVAILABLE]

Protocol distinction (never mix corpora):
  - Session-2 analysis uses the Session-2 balanced test / .npz preds above.
  - Local active protocol is identity_clean_v1 (separate corpus; ~4.5k imbalanced test).

CANONICAL DATASET PROTOCOL
  name              : identity_clean_v1
  strategy          : identity-disjoint
  seed              : 42
  dataset root      : /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/test_data_v3
  protocol dir      : /Users/pic

## E1. Canonical protocol and label semantics

In [21]:
proto = P.load_protocol(); meta = P.protocol_metadata()
checks = {
    "protocol is identity_clean_v1": proto.name == "identity_clean_v1",
    "strategy identity-disjoint": meta["split_strategy"] == "identity-disjoint",
    "seed == 42": meta["random_seed"] == 42,
    "labels 0=real, 1=fake": P.LABEL_MAPPING == {0: "real", 1: "fake"},
    "train == 20,991": meta["train_size"] == 20991,
    "val   == 4,498": meta["val_size"] == 4498,
    "test  == 4,499": meta["test_size"] == 4499,
    "sizes match CSVs": loader.split_sizes() == {"train": 20991, "val": 4498, "test": 4499},
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "canonical protocol drift detected"

  [PASS] protocol is identity_clean_v1
  [PASS] strategy identity-disjoint
  [PASS] seed == 42
  [PASS] labels 0=real, 1=fake
  [PASS] train == 20,991
  [PASS] val   == 4,498
  [PASS] test  == 4,499
  [PASS] sizes match CSVs


## E2. Dataset arithmetic and leakage

In [22]:
manifest = loader.load_manifest()
total = sum(loader.split_sizes().values())
removed = len(manifest) - total
print(f"manifest {len(manifest):,} - removed {removed:,} = protocol {total:,}  "
      f"-> {len(manifest)-removed == total}")

log = _ROOT / "experiments/results/eda_real_data/removed_exact_duplicates.csv"
if log.exists():
    n = len(pd.read_csv(log))
    print(f"audit log rows {n:,} == removed {removed:,} -> {n == removed}")
    print("Every removed duplicate is logged; nothing silently deleted.")

det = {s: loader.load_split(s, detailed=True) for s in P.SPLITS}
for key in ["identity", "video"]:
    sets = {s: set(d[key]) for s, d in det.items()}
    ov = {f"{a}-{b}": len(sets[a] & sets[b])
          for a, b in [("train", "val"), ("train", "test"), ("val", "test")]}
    tag = "PASS (must be 0)" if key == "identity" and sum(ov.values()) == 0 else (
          "KNOWN LIMITATION (not safe)" if key == "video" else "FAIL")
    print(f"{key:9s} overlap {ov}  -> {tag}")

manifest 30,691 - removed 703 = protocol 29,988  -> True
audit log rows 703 == removed 703 -> True
Every removed duplicate is logged; nothing silently deleted.
identity  overlap {'train-val': 0, 'train-test': 0, 'val-test': 0}  -> PASS (must be 0)
video     overlap {'train-val': 1509, 'train-test': 1542, 'val-test': 930}  -> KNOWN LIMITATION (not safe)


## E3. Evaluation populations are the right ones

In [23]:
PRED = CANONICAL_PREDICTIONS
preds = pd.read_csv(PRED)
try:
    loader.verify_matches_protocol(preds, "test")
    print(f"PASS: canonical evaluator predictions ({len(preds):,}) == canonical test split")
except ValueError as e:
    raise AssertionError(f"Canonical prediction handoff does not match protocol: {e}") from e

if S2.availability().eval_preds:
    c = S2.test_composition()
    print(f"PASS: Session-2 balanced test measured at n={c['n_total']:,} "
          f"({c['n_real']:,} real / {c['n_fake']:,} fake, 1:{c['real_fake_ratio']:.2f})")
    print("      matches the reference figure of 21,446 at 1:1")
print()
print("NOTE: two distinct corpora are in play. They are never merged:")
print("  - local identity_clean_v1 : 4,499-image test, ~96% fake -> use MCC")
print("  - Session-2 balanced      : 21,446-image test, 1:1     -> accuracy is fair")

PASS: canonical evaluator predictions (4,499) == canonical test split
PASS: Session-2 balanced test measured at n=21,446 (10,723 real / 10,723 fake, 1:1.00)
      matches the reference figure of 21,446 at 1:1

NOTE: two distinct corpora are in play. They are never merged:
  - local identity_clean_v1 : 4,499-image test, ~96% fake -> use MCC
  - Session-2 balanced      : 21,446-image test, 1:1     -> accuracy is fair


## E4. Headline results, each on its own corpus

In [24]:
import json as _json
mp = CANONICAL_EVAL_DIR / "metrics.json"
if not mp.exists():
    raise FileNotFoundError(f"Canonical evaluator metrics missing: {mp}")
m = _json.load(open(mp))
print("LOCAL identity_clean_v1 (imbalanced test - accuracy is misleading)")
print(f"   accuracy {m['accuracy']:.4f} | balanced acc {m['balanced_accuracy']:.4f} "
      f"| MCC {m['mcc']:.4f} | real recall {m['real_recall']:.4f}")
print(f"   TN/FP/FN/TP = {m['tn']}/{m['fp']}/{m['fn']}/{m['tp']}")

if S2.availability().eval_preds:
    print("\nSESSION-2 balanced test (1:1 - accuracy is fair)")
    display(S2.metrics_table()[["model", "acc%", "f1", "AUC", "real_acc",
                                "fake_recall", "FP", "FN"]].round(4))

LOCAL identity_clean_v1 (imbalanced test - accuracy is misleading)
   accuracy 0.8657 | balanced acc 0.5679 | MCC 0.0815 | real recall 0.2456
   TN/FP/FN/TP = 42/129/475/3853

SESSION-2 balanced test (1:1 - accuracy is fair)


,model,acc%,f1,AUC,real_acc,fake_recall,FP,FN
0,ViT-Plus A0 (old sampler),97.9110,0.9791,0.9979,0.9805,0.9777,209,239
1,ViT-Plus A1 (weak_family),98.4706,0.9848,0.9986,0.9792,0.9902,223,105
2,ConvNeXt (finetuned ref),99.2166,0.9921,0.9998,0.9979,0.9864,22,146
3,ViT-S/16 (session-1 ref),97.2349,0.9724,0.9960,0.9707,0.9740,314,279
4,Pretr ViT-Plus (frozen probe),89.4666,0.8955,0.9624,0.8869,0.9025,1213,1046
5,Pretr ConvNeXt (frozen probe),87.7739,0.8748,0.9553,0.9011,0.8544,1061,1561


## E5. Reference findings: reproduced, represented, or unavailable

In [25]:
av = S2.availability()
rows = [
    ("1. Train composition 123,582 / 94,025 fake / 29,557 real", "REFERENCE only",
     "raw Session-2 train CSV absent locally"),
    ("2. 9 real sources, diversity beyond FF++/Celeb-DF", "REFERENCE only",
     "shown in nb00 A2, clearly labelled"),
    ("3. 42 methods, long-tail pools", "REFERENCE only",
     "shown in nb00 A3; weak family PARSED from sampler"),
    ("4. Dimensions 256 / 512 / 178x218", "PARTIAL",
     "local dims MEASURED in nb00 A4; Session-2 dims REFERENCE"),
    ("5. Identity ~2.5-3.4 img/id, train-val identity overlap 0", "PARTIAL",
     "Session-2 counts REFERENCE; local identity+leakage MEASURED"),
    ("6. Splits: train/val/test-full/test-balanced", "PARTIAL",
     "balanced test 21,446 @1:1 MEASURED from predictions"),
    ("7. Sampler exposure A0 vs A1 (dfs 0.056 -> 0.52, x9)", "REPRODUCED",
     "recomputed with the project sampler rules, matches reference"),
    ("8. A1 vs A0 McNemar p=3.6e-10, fake p=3e-21, real p=0.31", "REPRODUCED",
     "computed from local paired predictions"),
    ("9. Per-method gains concentrated on the 8 weak methods", "REPRODUCED",
     "nb02_error_analysis C2/C3"),
    ("10. Pretrained probe ~89.5/87.8% vs finetuned ~98.5/99.2%", "REPRODUCED",
     "nb02_error_analysis C6"),
    ("11. Probes weakest on Face Swap / weak family", "REPRODUCED",
     "nb02_error_analysis C6"),
    ("12. Real-source difficulty (ff++_real hardest)", "REPRODUCED",
     "nb02_error_analysis C4"),
]
cov = pd.DataFrame(rows, columns=["reference finding", "status", "where / why"])
display(cov)
print(cov["status"].value_counts().to_string())
print("\nNo reference finding is presented as measured when it is not.")

,reference finding,status,where / why
0,"1. Train composition 123,582 / 94,025 fake / 2...",REFERENCE only,raw Session-2 train CSV absent locally
1,"2. 9 real sources, diversity beyond FF++/Celeb-DF",REFERENCE only,"shown in nb00 A2, clearly labelled"
2,"3. 42 methods, long-tail pools",REFERENCE only,shown in nb00 A3; weak family PARSED from sampler
3,4. Dimensions 256 / 512 / 178x218,PARTIAL,local dims MEASURED in nb00 A4; Session-2 dims...
4,"5. Identity ~2.5-3.4 img/id, train-val identit...",PARTIAL,Session-2 counts REFERENCE; local identity+lea...
5,6. Splits: train/val/test-full/test-balanced,PARTIAL,"balanced test 21,446 @1:1 MEASURED from predic..."
6,7. Sampler exposure A0 vs A1 (dfs 0.056 -> 0.5...,REPRODUCED,"recomputed with the project sampler rules, mat..."
7,"8. A1 vs A0 McNemar p=3.6e-10, fake p=3e-21, r...",REPRODUCED,computed from local paired predictions
8,9. Per-method gains concentrated on the 8 weak...,REPRODUCED,nb02_error_analysis C2/C3
9,10. Pretrained probe ~89.5/87.8% vs finetuned ...,REPRODUCED,nb02_error_analysis C6


status
REPRODUCED        6
REFERENCE only    3
PARTIAL           3

No reference finding is presented as measured when it is not.


## E6. Consistency verdict

* One canonical protocol (`identity_clean_v1`), one label mapping
  (`0=real, 1=fake`), one preprocessing (256x256) across all five notebooks.
* Split sizes reconcile with the manifest and the duplicate audit log.
* Identity and exact-duplicate cross-split overlap are 0.
* Video overlap and un-removed near-duplicates remain **documented limitations**
  and are never described as safe.
* Val/test are never rebalanced; balancing is training-side only.
* The two corpora are reported separately and never mixed.

**Known limitations carried forward**

1. Video/source overlap in `identity_clean_v1`.
2. 4,215 cross-split near-duplicate groups, quantified but not removed.
3. Session-2 raw training corpus absent locally - train-side composition is
   reference-only.
4. Local baseline is a 1-epoch-derived 8-epoch run whose MCC was still rising:
   not converged.